# Final figures and paper assembly

This notebook reads only **frozen analysis outputs** and generates manuscript figures. It performs no new discovery or hypothesis testing. Run after analysis folders 01-07 are complete.

Cross-platform effect magnitudes are not treated as directly comparable. Integrated panels therefore separate within-dataset quantitative effects from directional/concordance summaries.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Prefer the compact frozen-results bundle shipped with this repository.
# Fall back to the original Google Drive project when running in Colab.
candidates = [
    Path.cwd()/"frozen_results",
    Path.cwd().parent/"frozen_results",
]
ROOT = next((x for x in candidates if (x/"signatures/GSE328966_LOCKED_discovery_signatures.json").exists()), None)

if ROOT is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")
    except Exception as e:
        raise RuntimeError("Could not find repository frozen_results/ and Google Drive fallback is unavailable.") from e

FIG = (ROOT.parent/"08_Final_Figures") if ROOT.name == "frozen_results" else ROOT/"08_Final_Figures"
FIG.mkdir(exist_ok=True)

with open(ROOT/"signatures/GSE328966_LOCKED_discovery_signatures.json") as f:
    sig = json.load(f)

def savefig(fig, stem):
    fig.savefig(FIG/f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(FIG/f"{stem}.svg", bbox_inches="tight")
    fig.savefig(FIG/f"{stem}.png", dpi=300, bbox_inches="tight")

print("Using frozen results root:", ROOT)
print("Figure output:", FIG)


## Figure 1 — Definition and temporal collapse of the early Müller program

In [ ]:
disc = pd.read_csv(ROOT/"01_GSE328966_early_mouse/GSE328966_Muller_early_late_trajectory.csv")
disc = disc.rename(columns={"Unnamed: 0":"gene"})
ora = pd.read_csv(ROOT/"01_GSE328966_early_mouse/GSE328966_Muller_trajectory_LOCAL_ORA_FINAL.csv")

locked = {
    "Early184": sig["early_up_transient_all"],
    "IFN6": sig["early_IFN_alpha_beta"],
    "Retinoid5": sig["early_retinoid_metabolism"],
    "ECM13": sig["early_ECM_GAG_exploratory"]
}

traj_counts = pd.Series({
    "early_up_transient": len(sig["early_up_transient_all"]),
    "persistent_up": len(sig["persistent_up_all_descriptive"]),
    "up_to_down_reversal": len(sig["up_to_down_reversal_all_descriptive"])
})
show_classes = ["early_up_transient","persistent_up","up_to_down_reversal"]

prog=[]
for name,genes in locked.items():
    x=disc[disc.gene.isin(genes)]
    prog.append([name, x.early_effect.median(), x.late_effect.median()])
prog=pd.DataFrame(prog,columns=["signature","5D","15D"]).set_index("signature")

fig, ax = plt.subplots(1,3, figsize=(13,3.8))
ax[0].bar(range(len(show_classes)), [traj_counts.get(x,0) for x in show_classes])
ax[0].set_xticks(range(len(show_classes)), ["Early-up\ntransient","Persistent\nup","Up→down\nreversal"])
ax[0].set_ylabel("Genes")
ax[0].set_title("Locked early trajectories")

x=np.arange(len(prog)); w=.36
ax[1].bar(x-w/2,prog["5D"],w,label="5D")
ax[1].bar(x+w/2,prog["15D"],w,label="15D")
ax[1].axhline(0,lw=.8)
ax[1].set_xticks(x,prog.index,rotation=30,ha="right")
ax[1].set_ylabel("Median gene effect")
ax[1].set_title("Early activation collapses by 15D")
ax[1].legend(frameon=False)

qcol = "Adjusted P-value" if "Adjusted P-value" in ora.columns else [c for c in ora.columns if "adjust" in c.lower()][0]
sub = ora[(ora["trajectory_class"]=="early_up_transient") & (ora["library"].astype(str).str.contains("Reactome"))].copy()
sub = sub.sort_values(qcol).head(6).iloc[::-1]
ax[2].barh(range(len(sub)), -np.log10(sub[qcol].clip(lower=1e-12)))
ax[2].set_yticks(range(len(sub)), [str(x).split(" R-HSA")[0][:38] for x in sub["Term"]])
ax[2].set_xlabel("-log10 FDR")
ax[2].set_title("Early-up Reactome enrichment")

fig.suptitle("Figure 1. A transient resilience-associated Müller program emerges early in diabetes",y=1.03)
plt.tight_layout(); savefig(fig,"Figure1_Discovery"); plt.show()


## Figure 2 — Independent mouse progression and Müller/whole-retina divergence

In [ ]:
mb = pd.read_csv(ROOT/"04_GSE313176_mouse_longitudinal/GSE313176_LOCKED_signature_validation_exact_permutation.csv")
mm = pd.read_csv(ROOT/"04_GSE313176_mouse_longitudinal/scRNA_3mo/GSE313176_PRIMARY_Muller_locked_signature_summary.csv")
conc = pd.read_csv(ROOT/"06_CrossSpecies_Trajectory_Integration/Early184_genelevel_Spearman_tests.csv")

name_map={"early_up_transient_all":"Early184","early_IFN_alpha_beta":"IFN6","early_retinoid_metabolism":"Retinoid5","early_ECM_GAG_exploratory":"ECM13"}
mb["signature"]=mb.signature.map(lambda x:name_map.get(x,x))
mm["signature"]=mm.signature.map(lambda x:name_map.get(x,x))
keep=["Early184","IFN6","Retinoid5","ECM13"]

fig,ax=plt.subplots(1,3,figsize=(13,3.8))
for s in keep:
    z=mb[mb.signature==s].set_index("timepoint").reindex(["1mo","3mo","6mo"])
    ax[0].plot([1,3,6],z.delta_STZ_minus_Sham,marker="o",label=s)
ax[0].axhline(0,lw=.8); ax[0].set_xticks([1,3,6]); ax[0].set_xlabel("Months of diabetes"); ax[0].set_ylabel("STZ - Sham signature score"); ax[0].set_title("Replicated whole-retina bulk"); ax[0].legend(frameon=False,fontsize=8)

z=mm[mm.signature.isin(keep)].set_index("signature").reindex(keep)
ax[1].bar(range(4),z.median_log2FC)
ax[1].axhline(0,lw=.8); ax[1].set_xticks(range(4),keep,rotation=30,ha="right"); ax[1].set_ylabel("Median log2FC"); ax[1].set_title("3-month Müller scRNA (descriptive)")

keys=[("bulk_1mo","bulk_3mo"),("bulk_1mo","bulk_6mo"),("bulk_3mo","bulk_6mo"),("bulk_1mo","muller3mo"),("bulk_3mo","muller3mo"),("bulk_6mo","muller3mo")]
vals=[]
for a,b in keys:
    r=conc[(conc.context_A==a)&(conc.context_B==b)]
    if r.empty: r=conc[(conc.context_A==b)&(conc.context_B==a)]
    vals.append(float(r.rho.iloc[0]))
labels=["1m-3m","1m-6m","3m-6m","1m-MG","3m-MG","6m-MG"]
ax[2].bar(range(6),vals); ax[2].axhline(0,lw=.8); ax[2].set_xticks(range(6),labels,rotation=40,ha="right"); ax[2].set_ylabel("Spearman ρ within Early184"); ax[2].set_title("Whole retina is coherent; Müller diverges")

fig.suptitle("Figure 2. Chronic diabetic remodeling separates Müller and whole-retina trajectories",y=1.03)
plt.tight_layout(); savefig(fig,"Figure2_MouseProgression"); plt.show()


## Figure 3 — Human translation is heterogeneous and region/stage dependent

In [ ]:
hs = pd.read_csv(ROOT/"02_Human_scRNA_2026/Human_Muller_LOCKED_signature_donor_scores.csv")
hb = pd.read_csv(ROOT/"03_GSE160306_human_validation/GSE160306_FINAL_locked_signature_tests_79samples.csv")

fig,ax=plt.subplots(1,2,figsize=(12,4))
order=["NON","DM","DR"]
for j,g in enumerate(order):
    y=hs.loc[hs.status==g,"Early184"].to_numpy()
    ax[0].scatter(np.full(len(y),j),y,s=35)
    ax[0].plot([j-.18,j+.18],[np.mean(y),np.mean(y)],lw=2)
ax[0].axhline(0,lw=.8); ax[0].set_xticks(range(3),order); ax[0].set_ylabel("Donor-level Early184 score"); ax[0].set_title("Human Müller atlas")

mapn={"Early166":"Early184","IFN6":"IFN6","Retinoid5":"Retinoid5","ECM13":"ECM13"}
hb["signature"]=hb.signature.map(mapn)
cols=["DMvsCTRL_P","DMvsCTRL_M","NPDRvsCTRL_P","NPDRvsCTRL_M","ADVvsCTRL_P","ADVvsCTRL_M"]
H=hb[hb.contrast.isin(cols)].pivot(index="signature",columns="contrast",values="effect_meanZ").reindex(keep)[cols]
lim=np.nanmax(np.abs(H.values))
im=ax[1].imshow(H.values,aspect="auto",cmap="RdBu_r",vmin=-lim,vmax=lim)
ax[1].set_yticks(range(4),keep); ax[1].set_xticks(range(len(cols)),[c.replace("vsCTRL_","\n") for c in cols],rotation=35,ha="right"); ax[1].set_title("Human bulk retina: stage × region")
fig.colorbar(im,ax=ax[1],label="Mean-Z contrast")
fig.suptitle("Figure 3. Human translation is partial and context dependent",y=1.03)
plt.tight_layout(); savefig(fig,"Figure3_HumanTranslation"); plt.show()


## Figure 4 — Controlled human organoid perturbation does not reproduce the locked program

In [ ]:
od = pd.read_csv(ROOT/"05_GSE290024_organoid/GSE290024_direct_signature_tests.csv")
de = pd.read_csv(ROOT/"05_GSE290024_organoid/GSE290024_edgeR_DE_all_genes.csv")

fig,ax=plt.subplots(1,2,figsize=(10,3.8))
z=od.set_index("signature").reindex(keep)
ax[0].bar(range(4),z.effect); ax[0].axhline(0,lw=.8); ax[0].set_xticks(range(4),keep,rotation=30,ha="right"); ax[0].set_ylabel("Direct standardized score effect"); ax[0].set_title("Locked modules")

genes=["SAMHD1","GPC1","GPC3","COL9A2"]
g=de[de.symbol.isin(genes)].drop_duplicates("symbol").set_index("symbol").reindex(genes)
ax[1].bar(range(len(g)),g.logFC); ax[1].axhline(0,lw=.8); ax[1].set_xticks(range(len(g)),genes); ax[1].set_ylabel("log2FC, 25 mM vs 17-17.5 mM"); ax[1].set_title("Selective component remodeling")
fig.suptitle("Figure 4. Strong human retinal perturbation without coherent Early184 replication",y=1.03)
plt.tight_layout(); savefig(fig,"Figure4_Organoid"); plt.show()


## Figure 5 — Cross-system state transition

In [ ]:
traj = pd.read_csv(ROOT/"06_CrossSpecies_Trajectory_Integration/Locked_signature_cross_system_trajectory.csv")
M = pd.read_csv(ROOT/"06_CrossSpecies_Trajectory_Integration/Early184_cross_system_gene_effects_MASTER.csv")

rows=keep
primary=[
"Mouse bulk | 1mo","Mouse bulk | 3mo","Mouse Müller scRNA | 3mo","Mouse bulk | 6mo",
"Human Müller scRNA | DM-NON","Human Müller scRNA | DM-DR","Human organoid | 25mM-17mM",
"Human bulk retina | DMvsCTRL_P","Human bulk retina | DMvsCTRL_M",
"Human bulk retina | NPDRvsCTRL_P","Human bulk retina | NPDRvsCTRL_M",
"Human bulk retina | ADVvsCTRL_P","Human bulk retina | ADVvsCTRL_M"]

T=traj.copy(); T["context"]=T.system+" | "+T.contrast
B=T[T.context.isin(primary)].pivot(index="signature",columns="context",values="effect").reindex(rows)[primary]
D=np.sign(B)
A=T[(T.system=="Mouse discovery Müller")].pivot(index="signature",columns="contrast",values="effect").reindex(rows)[["5D","15D"]]

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(15,3.5),gridspec_kw={"width_ratios":[2,13]})
lim=np.nanmax(np.abs(A.values)); ax1.imshow(A.values,cmap="RdBu_r",aspect="auto",vmin=-lim,vmax=lim)
for i in range(4):
    for j in range(2): ax1.text(j,i,f"{A.iloc[i,j]:.2f}",ha="center",va="center",fontsize=8)
ax1.set_xticks([0,1],["5D","15D"]); ax1.set_yticks(range(4),rows); ax1.set_title("Discovery Müller")
ax2.imshow(D.values,cmap="RdBu_r",vmin=-1,vmax=1,aspect="auto")
ax2.set_xticks(range(len(primary)),[x.replace(" | ","\n") for x in primary],rotation=55,ha="right",fontsize=8); ax2.set_yticks(range(4),[""]*4); ax2.set_title("Independent systems — direction only")
fig.suptitle("Figure 5A. Early Müller activation transitions into context-dependent remodeling",y=1.04)
plt.tight_layout(); savefig(fig,"Figure5A_CrossSystemDirection"); plt.show()

contexts=["late_effect","bulk_1mo","bulk_3mo","muller3mo","bulk_6mo","organoid","human_DM","human_DR"]
labels=["Mouse Müller 15D","Mouse bulk 1mo","Mouse bulk 3mo","Mouse Müller 3mo","Mouse bulk 6mo","Human organoid","Human Müller DM","Human Müller DR"]
rhos=[]
for c in contexts:
    x=M[["early_effect",c]].dropna(); rhos.append(spearmanr(x.early_effect,x[c]).statistic)
fig,ax=plt.subplots(figsize=(7,4.5)); ax.barh(range(len(rhos)),rhos); ax.axvline(0,lw=.8); ax.set_yticks(range(len(labels)),labels); ax.invert_yaxis(); ax.set_xlabel("Spearman ρ vs early 5D Müller effect"); ax.set_title("Limited preservation of the early Müller gene program across downstream systems")
plt.tight_layout(); savefig(fig,"Figure5B_GeneLevelConcordance"); plt.show()


## Supplementary — Lin 2025 rat Müller threshold-aware validation

In [ ]:
rat = pd.read_csv(ROOT/"07_Lin2025_Rat_Muller_Validation/Lin2025_all_locked_signatures_thresholded_tests.csv")
fig,ax=plt.subplots(figsize=(7,3.5))
z=rat[rat.signature=="Early184"].set_index("timepoint").reindex(["1M","2M","3M"])
ax.bar(range(3),z.median_log2FC); ax.axhline(0,lw=.8); ax.set_xticks(range(3),["1M","2M","3M"]); ax.set_ylabel("Median log2FC among reported Early184 genes"); ax.set_title("Lin 2025 rat Müller: thresholded Early184 trajectory")
for i,(n,up) in enumerate(zip(z.reported_n,z.fraction_up)):
    ax.text(i,z.median_log2FC.iloc[i],f"n={int(n)}; {up:.0%} up",ha="center",va="bottom" if z.median_log2FC.iloc[i]>=0 else "top",fontsize=8)
plt.tight_layout(); savefig(fig,"Supplement_Lin2025"); plt.show()
print("FINAL FIGURE NOTEBOOK COMPLETE")
